# SBERP candle exploration

Loads cached 1min and 5min SBERP candles and plots 1min candles with an EWM trend of close.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "bot")
from config import ASSETS, CACHE_DIR

In [ ]:
figi = ASSETS["SBERP"]

df_1m = pd.read_csv(CACHE_DIR / f"{figi}_1min.csv", parse_dates=["timestamp"])
df_5m = pd.read_csv(CACHE_DIR / f"{figi}_5min.csv", parse_dates=["timestamp"])

print(f"1min: {len(df_1m):>7,} bars  {df_1m.timestamp.min()} .. {df_1m.timestamp.max()}")
print(f"5min: {len(df_5m):>7,} bars  {df_5m.timestamp.min()} .. {df_5m.timestamp.max()}")
df_1m.head()

In [ ]:
# One day of 1min candles + EWM trend of close.
DAY = df_1m.timestamp.dt.date.iloc[-1]  # pick any cached date here
TREND_ALPHA = 0.02

# EWM over the full series so the trend has warmup before the displayed day.
trend = df_1m["close"].ewm(alpha=TREND_ALPHA).mean()

day = df_1m[df_1m.timestamp.dt.date == DAY]
up = day["close"] >= day["open"]
colors = np.where(up, "seagreen", "tomato")

fig, ax = plt.subplots(figsize=(16, 6))
ax.vlines(day.timestamp, day["low"], day["high"], color=colors, lw=0.5)
body_lo = np.minimum(day["open"], day["close"])
body_hi = np.maximum(day["open"], day["close"])
ax.vlines(day.timestamp, body_lo, body_hi, color=colors, lw=2.5)
ax.plot(day.timestamp, trend[day.index], color="orange", lw=1.2, label=f"ewm(close, alpha={TREND_ALPHA})")

ax.set_title(f"SBERP 1min candles — {DAY}")
ax.set_ylabel("Price (RUB)")
ax.legend(loc="upper left")
ax.grid(True, lw=0.3)
ax.margins(x=0)
plt.show()